# 实验：从编号、权限和版本到检索指标

> 状态：verified；教学语料 v1，8 条文档事件（含替换）与 8 条查询。

所有设备和步骤均为虚构教学数据，不是设备维修建议。模型调用为 0：词法 BM25 + 精确编码 + RRF + 抽取式引用。真实 embedding/reranker 接口存在但本次未运行。本文保留跨语言失败，避免把词法匹配当语义能力。正文：[混合检索](../01-concepts/02-hybrid-retrieval-and-reranking.md)。

In [1]:
from pathlib import Path
import sys,json,math
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
project = ROOT / "10-Knowledge/06-rag-and-knowledge-systems/05-code/rag-pipeline-python"
sys.path.insert(0,str(project/"src"))
from rag_pipeline import Index,load_jsonl,answer,verify_citation,rrf,retrieval_metrics,bm25
idx=Index()
docs=load_jsonl(project/"fixtures/corpus.jsonl")
for doc in docs: idx.upsert(doc)
queries=[json.loads(x) for x in (project/"fixtures/queries.jsonl").read_text().splitlines()]
print({"document_events":len(docs),"queries":len(queries),"embedding_executed":False})

{'document_events': 8, 'queries': 8, 'embedding_executed': False}


先看每个查询的文档排名。相关性标签用 2 表示直接支持、0/未标注表示不相关。报告以文档去重，Top-3 指三个不同文档；无答案查询的 recall/nDCG 不定义，单独检查是否为空。

In [2]:
rows=[]
for q in queries:
    filters={key:q[key] for key in ("tenant","product","version")}
    hits=idx.search(q["query"],**filters,k=3,unit="document")
    ranking=list(dict.fromkeys(h.chunk.doc_id for h in hits))
    rows.append({"id":q["id"],"ranking":ranking,**retrieval_metrics(ranking,q["relevance"],3)})
print(json.dumps(rows,ensure_ascii=False,indent=2))
assert rows[0]["ranking"]==["fan-guide"]
assert rows[1]["ranking"]==["voltage-guide"]
assert all(rows[i]["empty"] for i in (4,5,6))
assert rows[7]["recall"]==0

[
  {
    "id": "q-code",
    "ranking": [
      "fan-guide"
    ],
    "recall": 1.0,
    "mrr": 1.0,
    "ndcg": 1.0,
    "empty": false
  },
  {
    "id": "q-near-code",
    "ranking": [
      "voltage-guide"
    ],
    "recall": 1.0,
    "mrr": 1.0,
    "ndcg": 1.0,
    "empty": false
  },
  {
    "id": "q-cooling",
    "ranking": [
      "cooling",
      "voltage-guide",
      "release-notes"
    ],
    "recall": 1.0,
    "mrr": 1.0,
    "ndcg": 1.0,
    "empty": false
  },
  {
    "id": "q-timeout",
    "ranking": [
      "timeout"
    ],
    "recall": 1.0,
    "mrr": 1.0,
    "ndcg": 1.0,
    "empty": false
  },
  {
    "id": "q-no-answer",
    "ranking": [],
    "recall": null,
    "mrr": null,
    "ndcg": null,
    "empty": true
  },
  {
    "id": "q-old-version",
    "ranking": [],
    "recall": null,
    "mrr": null,
    "ndcg": null,
    "empty": true
  },
  {
    "id": "q-tenant",
    "ranking": [],
    "recall": null,
    "mrr": null,
    "ndcg": null,
    "empty": true
 

查询 `airflow obstruction` 在中文语料中词法零命中。它的 gold 是 cooling，但当前检索召回为 0；这说明需要跨语言语义模型或受控术语映射，不能宣称本实验验证了 Dense Hybrid 的收益。

In [3]:
for mode in ("bm25","exact","hybrid"):
    scores=[]
    for q in queries:
        if not q["relevance"]: continue
        hits=idx.search(q["query"],tenant=q["tenant"],product=q["product"],version=q["version"],mode=mode,k=3,unit="document")
        scores.append(retrieval_metrics([h.chunk.doc_id for h in hits],q["relevance"],3)["recall"])
    print(mode,"answerable_mean_recall@3",sum(scores)/len(scores))

bm25 answerable_mean_recall@3 0.8
exact answerable_mean_recall@3 0.4
hybrid answerable_mean_recall@3 0.8


本语料上 BM25 与 hybrid 可能相同，不能为了展示提升去修改标签或隐藏失败。Exact 通道适合编号；非编号查询没有 exact 候选。下面手算排名权重和有等级相关性的 nDCG。

In [4]:
bm25_value = bm25("fan", ["fan fan", "log step", "power off"])[0]
print("N=3, df=1, tf=2, 等长文档的 BM25:", bm25_value)
assert math.isclose(bm25_value, math.log(1 + 2.5 / 1.5) * 1.375)

fused=rrf([["a","b"],["b","c"]],constant=60)
print(fused)
assert max(fused,key=fused.get)=="b"
metrics=retrieval_metrics(["bad","good","okay"],{"good":2,"okay":1},3)
print(metrics)
assert math.isclose(metrics["ndcg"],(3/math.log2(3)+1/math.log2(4))/(3+1/math.log2(3)))

N=3, df=1, tf=2, 等长文档的 BM25: 1.3486402228911236
{'a': 0.01639344262295082, 'b': 0.03252247488101534, 'c': 0.016129032258064516}
{'recall': 1.0, 'mrr': 0.5, 'ndcg': 0.6590018048024133, 'empty': False}


In [5]:
result=answer(idx,"ALM-12003",tenant="alpha",product="DEMO-A",version="2")
print(result.text)
assert all(verify_citation(idx,c,tenant="alpha") for c in result.citations)
old=result.citations[0]
idx.delete("fan-guide",tenant="alpha")
assert not verify_citation(idx,old,tenant="alpha")
assert answer(idx,"ALM-12003",tenant="alpha",product="DEMO-A").abstained
print("删除后引用失效，alpha 范围不再返回对应证据。")

[1] 教学设备 DEMO-A；ALM-12003 风扇告警。v2 操作：读取转速日志，检查供电连接，确认原因后再处理；不得直接重启。此为虚构教学步骤，不是设备操作指南。
删除后引用失效，alpha 范围不再返回对应证据。


**可得结论**：限定的教学任务验证了编号、权限、版本、删除、RRF、指标与引用完整性。**不可得结论**：任何真实模型/企业语料效果，PDF 解析、开放生成的 groundedness，以及生产并发持久化。工程运行方法见 [README](../05-code/rag-pipeline-python/README.md)。

下一步读[另一组英文教学题的真实模型检索结果](../../../20-Projects/learning-workbench/artifacts/real-models/retrieval.json)。先只比较那一组内部的四种模式，不能把本文的中文基线分数与另一组分数直接相减。